In [18]:
import pandas as pd
import numpy as np
from calendar import monthrange
from sqlalchemy import create_engine, text
from pandas.tseries.offsets import MonthEnd

def create_date_column(df):
    # 각 월의 마지막 날짜 계산 함수
    def get_last_day_of_month(year, month):
        return monthrange(int(year), int(month))[1]

    # 날짜 열 생성
    df['Date'] = df.apply(
        lambda row: pd.to_datetime(
            f"{int(row['회계연도'])}-{int(row['결산월'])}-{get_last_day_of_month(row['회계연도'], row['결산월'])}"
        ),
        axis=1
    )

    # 주기가 분기형이면, 날짜를 해당 분기 말일로 직접 지정
    if '주기' in df.columns:
        quarter_map = {
            '1Q': '-03-31',
            '2Q': '-06-30',
            '3Q': '-09-30',
            '4Q': '-12-31'
        }

        def override_to_quarter_end(row):
            q = str(row['주기']).strip()
            y = int(row['회계연도'])
            return pd.to_datetime(f"{y}{quarter_map[q]}") if q in quarter_map else row['Date']

        df['Date'] = df.apply(override_to_quarter_end, axis=1)

    return df

# ======================
# 1. 데이터 로드 및 전처리
# ======================
#  C:\Users\82108\OneDrive\바탕 화면\investment\data\raw_data : 여기로 경로 이동 함
path = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\raw_data\is_bs_cf_Dataguide_2025_4Q.xlsx"
sheet_name = "KOSPI"

raw_df = pd.read_excel(path, sheet_name=sheet_name)

# 기준 행
row_header_1 = 8  # 항목명
row_header_2 = 9  # 단위, 코드, 분류 등

# 복합 컬럼명 생성
def combine_headers(col):
    item = str(raw_df.loc[row_header_1, col]) if pd.notna(raw_df.loc[row_header_1, col]) else ""
    unit = str(raw_df.loc[row_header_2, col]) if pd.notna(raw_df.loc[row_header_2, col]) else ""
    combined = f"{unit.strip()}: {item.strip()}" if unit and item else item or unit
    return combined if combined else col

# 새로운 컬럼 리스트 생성
new_columns = [combine_headers(col) for col in raw_df.columns]

# 컬럼명 적용
df_cleaned = raw_df.copy()
df_cleaned.columns = new_columns
df_cleaned = df_cleaned.iloc[10:].reset_index(drop=True)

# "Name"을 "company_name"으로 변경
if "Name" in df_cleaned.columns:
    df_cleaned = df_cleaned.rename(columns={"Name": "company_name"})

print("=== 전처리 후 컬럼 목록 (처음 30개) ===")
for i, col in enumerate(df_cleaned.columns[:30], 1):
    print(f"{i}. {col}")

# ======================
# 2. 실제 컬럼명 패턴 분석
# ======================
print("\n=== Symbol/Code 관련 컬럼 검색 ===")
for col in df_cleaned.columns:
    col_lower = str(col).lower()
    if any(keyword in col_lower for keyword in ['symbol', 'code', '코드', '종목']):
        print(f"  - {col}")

print("\n=== 년도 관련 컬럼 검색 ===")
for col in df_cleaned.columns:
    col_lower = str(col).lower()
    if any(keyword in col_lower for keyword in ['year', '년', '연도', '회계']):
        print(f"  - {col}")

print("\n=== 분기/주기 관련 컬럼 검색 ===")
for col in df_cleaned.columns:
    col_lower = str(col).lower()
    if any(keyword in col_lower for keyword in ['quarter', 'q', '분기', '주기', 'period']):
        print(f"  - {col}")

# ======================
# 3. 수동으로 컬럼 지정 (위 출력 결과를 보고 수정)
# ======================
# 실제 컬럼명을 확인한 후 직접 지정
col_symbol = 'Symbol'  # 실제 컬럼명으로 수정 필요
col_year = None  # 위 출력 결과를 보고 수정
col_period = None  # 위 출력 결과를 보고 수정

# 아래는 자동 검색 시도
for col in df_cleaned.columns:
    # Symbol 찾기
    if col_symbol == 'Symbol' and col in df_cleaned.columns:
        col_symbol = col
        break
    elif 'Symbol' in str(col) or 'CODE' in str(col).upper():
        col_symbol = col
        break

# 회계년 찾기 (더 관대한 조건)
for col in df_cleaned.columns:
    col_str = str(col)
    if '년' in col_str and '천원' not in col_str:
        col_year = col
        break

# 주기 찾기 (더 관대한 조건)
for col in df_cleaned.columns:
    col_str = str(col)
    if ('분기' in col_str or '주기' in col_str or 'Q' in col_str) and '천원' not in col_str:
        col_period = col
        break

# 재무 지표 컬럼 찾기
def find_column(df, keyword):
    for col in df.columns:
        if keyword in str(col) and '(천원)' in str(col):
            return col
    return None

col_sales = find_column(df_cleaned, '매출액')
col_gross = find_column(df_cleaned, '매출총이익')
col_operating = find_column(df_cleaned, '영업이익')
col_continuing = find_column(df_cleaned, '계속사업이익')
col_net_income = find_column(df_cleaned, '당기순이익')
col_noncurrent_liab = find_column(df_cleaned, '비유동부채')

# 자본 컬럼 찾기
col_equity = None
for col in df_cleaned.columns:
    if "자본총계" in str(col) and "지배" in str(col) and "(천원)" in str(col):
        col_equity = col
        break

# ======================
# 4. 결과 확인
# ======================
print("\n=== 컬럼 매칭 결과 ===")
col_mapping = {
    '종목코드': col_symbol,
    '회계년': col_year,
    '주기': col_period,
    '매출액': col_sales,
    '매출총이익': col_gross,
    '영업이익': col_operating,
    '계속사업이익': col_continuing,
    '당기순이익': col_net_income,
    '비유동부채': col_noncurrent_liab,
    '자본': col_equity
}

for name, col in col_mapping.items():
    status = "✓" if col is not None else "✗"
    print(f"{status} {name}: {col}")

# Missing 체크
missing_cols = [name for name, col in col_mapping.items() if col is None]

if missing_cols:
    print(f"\n⚠️ 다음 컬럼을 찾지 못했습니다: {', '.join(missing_cols)}")
    print("\n수동으로 컬럼명을 확인하고 코드에서 직접 지정해주세요.")
    print("예: col_year = '실제컬럼명'")

    # 일단 계속 진행 (선택적)
    input("\n계속하려면 Enter를 누르세요 (에러 발생 가능)...")

# ======================
# 5. 숫자형으로 변환
# ======================
numeric_cols = [col_sales, col_gross, col_operating, col_continuing,
                col_net_income, col_noncurrent_liab, col_equity]
numeric_cols = [col for col in numeric_cols if col is not None]

for col in numeric_cols:
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# ======================
# 6. 데이터 정렬 및 YoY 계산
# ======================
if col_symbol and col_year and col_period:
    df_cleaned = df_cleaned.sort_values(by=[col_symbol, col_year, col_period])

    # YoY 증가율 계산
    if col_sales:
        df_cleaned["YoY_매출액"] = df_cleaned.groupby(col_symbol)[col_sales].pct_change(periods=4, fill_method=None)
    if col_gross:
        df_cleaned["YoY_매출총이익"] = df_cleaned.groupby(col_symbol)[col_gross].pct_change(periods=4, fill_method=None)
    if col_operating:
        df_cleaned["YoY_영업이익"] = df_cleaned.groupby(col_symbol)[col_operating].pct_change(periods=4, fill_method=None)
    if col_continuing:
        df_cleaned["YoY_계속사업이익"] = df_cleaned.groupby(col_symbol)[col_continuing].pct_change(periods=4, fill_method=None)
    if col_net_income:
        df_cleaned["YoY_당기순이익"] = df_cleaned.groupby(col_symbol)[col_net_income].pct_change(periods=4, fill_method=None)
else:
    print("\n⚠️ 정렬 및 YoY 계산을 건너뜁니다 (필수 컬럼 부족)")

# ======================
# 7. 수익성 비율 계산
# ======================
if col_gross and col_sales:
    df_cleaned["매출총이익률"] = df_cleaned[col_gross] / df_cleaned[col_sales]
if col_operating and col_sales:
    df_cleaned["영업이익률"] = df_cleaned[col_operating] / df_cleaned[col_sales]
if col_net_income and col_sales:
    df_cleaned["순이익률"] = df_cleaned[col_net_income] / df_cleaned[col_sales]
if col_noncurrent_liab and col_equity:
    df_cleaned["비유동부채비율"] = df_cleaned[col_noncurrent_liab] / df_cleaned[col_equity]

# ======================
# 8. 결과 미리보기
# ======================
display_cols = []
if col_symbol:
    display_cols.append(col_symbol)
if col_year:
    display_cols.append(col_year)
if col_period:
    display_cols.append(col_period)
if col_sales:
    display_cols.append(col_sales)

display_cols += [col for col in ["YoY_매출액", "YoY_영업이익", "영업이익률", "순이익률", "비유동부채비율"]
                 if col in df_cleaned.columns]

if display_cols:
    print("\n=== 결과 미리보기 ===")
    print(df_cleaned[display_cols].head(20))

    print("\n=== 통계 정보 ===")
    print(f"총 행 수: {len(df_cleaned)}")
    if col_symbol:
        print(f"고유 종목 수: {df_cleaned[col_symbol].nunique()}")

=== 전처리 후 컬럼 목록 (처음 30개) ===
1. 코드
2. 코드명
3. 결산월
4. 회계연도
5. 분기
6. 매출액(천원)
7. 매출총이익(천원)
8. 영업이익(천원)
9. 계속사업이익(천원)
10. 당기순이익(천원)
11. 총포괄이익(지배)(천원)
12. 유형자산감가상각비(천원)
13. 연구개발비(천원)
14. 자산총계(천원)
15. 유동자산(천원)
16. 당좌자산(천원)
17. 매출채권및기타채권(천원)
18. 재고자산(천원)
19. 투자부동산(천원)
20. 비유동부채(천원)
21. 자본총계(천원)
22. 자본총계(지배)(천원)
23. 영업활동으로인한현금흐름(천원)
24. 영업활동으로인한현금흐름(TTM)(천원)
25. 영업활동으로인한현금흐름(평균)(천원)
26. 배당금지급(영업,투자,재무)(천원)

=== Symbol/Code 관련 컬럼 검색 ===
  - 코드
  - 코드명

=== 년도 관련 컬럼 검색 ===
  - 회계연도

=== 분기/주기 관련 컬럼 검색 ===
  - 분기

=== 컬럼 매칭 결과 ===
✓ 종목코드: 코드
✗ 회계년: None
✓ 주기: 분기
✓ 매출액: 매출액(천원)
✓ 매출총이익: 매출총이익(천원)
✓ 영업이익: 영업이익(천원)
✓ 계속사업이익: 계속사업이익(천원)
✓ 당기순이익: 당기순이익(천원)
✓ 비유동부채: 비유동부채(천원)
✓ 자본: 자본총계(지배)(천원)

⚠️ 다음 컬럼을 찾지 못했습니다: 회계년

수동으로 컬럼명을 확인하고 코드에서 직접 지정해주세요.
예: col_year = '실제컬럼명'

⚠️ 정렬 및 YoY 계산을 건너뜁니다 (필수 컬럼 부족)

=== 결과 미리보기 ===
         코드  분기       매출액(천원)     영업이익률      순이익률   비유동부채비율
0   A005930  1Q  1.441363e+10  0.278131  0.217765  0.020830
1   A005930  2Q  1.497945e+10  0.249211  0.209163  0.022165
2   

In [22]:
df_cleaned = df_cleaned.dropna(subset=['회계연도', '결산월'])

In [23]:
df_cleaned = create_date_column(df_cleaned)

In [26]:
df_cleaned

,코드,코드명,결산월,회계연도,분기,매출액(천원),매출총이익(천원),영업이익(천원),계속사업이익(천원),당기순이익(천원),...,자본총계(지배)(천원),영업활동으로인한현금흐름(천원),영업활동으로인한현금흐름(TTM)(천원),영업활동으로인한현금흐름(평균)(천원),"배당금지급(영업,투자,재무)(천원)",매출총이익률,영업이익률,순이익률,비유동부채비율,Date
0,A005930,삼성전자,12,2004,1Q,1.441363e+10,5.696279e+09,4.008878e+09,3.138788e+09,3.138788e+09,...,3.177934e+10,4531179000,14145215000,4233582000,-805143000,0.395201,0.278131,0.217765,0.020830,2004-12-31
1,A005930,삼성전자,12,2004,2Q,1.497945e+10,5.646279e+09,3.733037e+09,3.133150e+09,3.133150e+09,...,3.280358e+10,4107685000,16038393000,4319432000,NaN,0.376935,0.249211,0.209163,0.022165,2004-12-31
2,A005930,삼성전자,12,2004,3Q,1.434394e+10,4.827464e+09,2.742345e+09,2.689464e+09,2.689464e+09,...,3.454045e+10,3318266000,15893115000,3712975500,-791139000,0.336551,0.191185,0.187498,0.023215,2004-12-31
3,A005930,삼성전자,12,2004,4Q,1.389533e+10,4.182651e+09,1.532617e+09,1.825340e+09,1.825340e+09,...,3.444041e+10,2847215000,14804345000,3082740500,1000,0.301011,0.110297,0.131364,0.019025,2004-12-31
4,A005930,삼성전자,12,2005,1Q,1.381218e+10,4.136289e+09,2.149925e+09,1.498412e+09,1.498412e+09,...,3.519619e+10,2790733000,13063899000,2818974000,-772711000,0.299467,0.155654,0.108485,0.048231,2005-12-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69867,A001140,국보,12,2024,4Q,8.304650e+06,4.367754e+04,-1.157682e+06,-1.655430e+07,-1.655430e+07,...,9.811438e+06,-1079892.82,-368298.32,-1011797.75,NaN,0.005259,-0.139402,-1.993378,0.464127,2024-12-31
69868,A001140,국보,12,2025,1Q,1.410657e+06,-6.187572e+05,-2.022279e+06,-4.763896e+06,-4.763896e+06,...,5.043219e+06,-419974.76,-4495857.34,-749933.79,NaN,-0.438631,-1.433573,-3.377077,0.950267,2025-12-31
69869,A001140,국보,12,2025,2Q,9.607326e+05,-1.123679e+05,-1.584083e+06,-3.204117e+06,-3.204117e+06,...,1.305313e+06,-51038.94,-2494609.2,-235506.85,NaN,-0.116961,-1.648828,-3.335077,2.714044,2025-12-31
69870,A001140,국보,12,2025,3Q,1.048771e+06,7.142752e+04,-6.727630e+05,-6.245824e+06,-6.245824e+06,...,-4.944810e+06,-4675342.34,-6226248.86,-2363190.64,NaN,0.068106,-0.641478,-5.955376,-0.650594,2025-12-31


In [41]:
def convert_to_long_format_v2(df_cleaned):
    """
    컬럼 인덱스로 접근하여 변환 + 안전한 정수형 데이터 처리
    """
    import numpy as np

    def create_date(year, quarter):
        quarter_end_months = {
            '1Q': '03-31', '2Q': '06-30',
            '3Q': '09-30', '4Q': '12-31'
        }
        return f"{year}-{quarter_end_months[quarter]}"

    # 날짜 생성
    if 'date' not in df_cleaned.columns:
        df_cleaned['date'] = df_cleaned.apply(
            lambda row: create_date(row['회계연도'], row['분기']),
            axis=1
        )

    # 작업용 데이터프레임 생성
    df_temp = df_cleaned.copy()

    # symbol과 company_name 컬럼 추가
    df_temp['symbol'] = df_temp.iloc[:, 0]
    df_temp['company_name'] = df_temp.iloc[:, 1]

    # 기본 컬럼
    base_cols = ['symbol', 'company_name', 'date']

    # 매출액 이후, Date/date 제외한 지표 컬럼들
    indicator_start_idx = df_cleaned.columns.get_loc('매출액(천원)')
    indicator_cols = [
        col for col in df_cleaned.columns[indicator_start_idx:]
        if col not in ['Date', 'date']
    ]

    print(f"변환할 지표 수: {len(indicator_cols)}")

    # Long format 변환
    df_long = pd.melt(
        df_temp[base_cols + indicator_cols],
        id_vars=base_cols,
        value_vars=indicator_cols,
        var_name='indicator',
        value_name='value'
    )

    # 안전한 정수 변환 함수
    def safe_int_convert(val):
        if pd.isna(val):
            return val
        try:
            if np.isfinite(val) and val == int(val):
                return int(val)
            return val
        except (ValueError, TypeError, OverflowError):
            return val

    # value 컬럼에 적용
    df_long['value'] = df_long['value'].apply(safe_int_convert)

    # 정렬
    df_long = df_long.sort_values(['symbol', 'date', 'indicator']).reset_index(drop=True)

    print(f"\n변환 완료: {len(df_long):,} rows")
    print(f"고유 종목: {df_long['symbol'].nunique()}")
    print(f"고유 날짜: {df_long['date'].nunique()}")
    print(f"고유 지표: {df_long['indicator'].nunique()}")

    return df_long


# 사용
df_long = convert_to_long_format_v2(df_cleaned)
print(df_long.tail(20))


변환할 지표 수: 25

변환 완료: 1,746,800 rows
고유 종목: 794
고유 날짜: 88
고유 지표: 25
          symbol company_name        date              indicator         value
1746780  A950210   프레스티지바이오파마  2025-12-31              매출총이익(천원) -2.143602e+07
1746781  A950210   프레스티지바이오파마  2025-12-31                 매출총이익률 -3.879041e+00
1746782  A950210   프레스티지바이오파마  2025-12-31    배당금지급(영업,투자,재무)(천원)           NaN
1746783  A950210   프레스티지바이오파마  2025-12-31              비유동부채(천원)  7.202161e+07
1746784  A950210   프레스티지바이오파마  2025-12-31                비유동부채비율  1.752308e-01
1746785  A950210   프레스티지바이오파마  2025-12-31                   순이익률 -1.645663e+00
1746786  A950210   프레스티지바이오파마  2025-12-31              연구개발비(천원)  1.912290e+07
1746787  A950210   프레스티지바이오파마  2025-12-31               영업이익(천원) -1.431410e+07
1746788  A950210   프레스티지바이오파마  2025-12-31                  영업이익률 -2.590266e+00
1746789  A950210   프레스티지바이오파마  2025-12-31  영업활동으로인한현금흐름(TTM)(천원) -5.213540e+07
1746790  A950210   프레스티지바이오파마  2025-12-31       영업활동으로인한현금흐름(천원)

In [28]:
# 사용
df_long = convert_to_long_format_v2(df_cleaned)

# inf → NaN, 그 뒤 객체형 변환 보정
df_long = df_long.replace([np.inf, -np.inf], np.nan).infer_objects(copy=False)

# 또는 특정 열만 안전하게 처리
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")
df_long = df_long.dropna(subset=["value"])

new_col = ['symbol', 'company_name', 'date', 'indicator', 'value']
df_long.columns = new_col

fs_value_df = df_long

변환할 지표 수: 25

변환 완료: 1,746,800 rows
고유 종목: 794
고유 날짜: 88
고유 지표: 25


In [37]:
def upload_fs_data_to_db(df_long, db_info, table_name="korea_fs_data_from_DG", chunk_size=1000):
    df = df_long.copy()

    # 컬럼명 표준화
    rename_map = {}
    if 'Symbol' in df.columns:
        rename_map['Symbol'] = 'ticker'
    elif 'symbol' in df.columns:
        rename_map['symbol'] = 'ticker'

    if 'Date' in df.columns:
        rename_map['Date'] = 'date'

    df = df.rename(columns=rename_map)

    required_cols = ['ticker', 'company_name', 'date', 'indicator', 'value']
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"필수 컬럼 누락: {missing_cols}")

    # 데이터 정리
    df['ticker'] = df['ticker'].astype(str).str.strip()
    df['company_name'] = df['company_name'].astype(str).str.strip()
    df['indicator'] = df['indicator'].astype(str).str.strip()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=['ticker', 'date', 'indicator'])
    df['date'] = df['date'].dt.date
    df = df.where(pd.notnull(df), None)

    df = df.drop_duplicates(subset=['ticker', 'date', 'indicator'], keep='last').reset_index(drop=True)

    print(f"업로드 대상 rows: {len(df):,}")

    conn = None
    cursor = None

    try:
        conn = pymysql.connect(
            host=db_info['host'],
            user=db_info['user'],
            password=db_info['password'],
            database=db_info['database'],
            port=db_info['port'],
            charset='utf8mb4',
            autocommit=False
        )
        cursor = conn.cursor()

        create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS `{table_name}` (
            `ticker` VARCHAR(20) NOT NULL,
            `company_name` VARCHAR(100),
            `date` DATE NOT NULL,
            `indicator` VARCHAR(100) NOT NULL,
            `value` DOUBLE,
            `created_at` DATETIME DEFAULT CURRENT_TIMESTAMP,
            `updated_at` DATETIME DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
            PRIMARY KEY (`ticker`, `date`, `indicator`)
        );
        """
        cursor.execute(create_table_sql)
        conn.commit()

        insert_sql = f"""
        INSERT INTO `{table_name}` (`ticker`, `company_name`, `date`, `indicator`, `value`)
        VALUES (%s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            `company_name` = VALUES(`company_name`),
            `value` = VALUES(`value`),
            `updated_at` = CURRENT_TIMESTAMP;
        """

        rows = df[['ticker', 'company_name', 'date', 'indicator', 'value']].values.tolist()

        for i in range(0, len(rows), chunk_size):
            chunk = rows[i:i + chunk_size]
            cursor.executemany(insert_sql, chunk)
            conn.commit()
            print(f"{min(i + chunk_size, len(rows)):,} / {len(rows):,} rows 완료")

        print(f"✅ 총 {len(df):,}개 row 업로드 완료")

    except Exception as e:
        if conn is not None:
            try:
                conn.rollback()
            except Exception:
                pass
        raise e

    finally:
        if cursor is not None:
            try:
                cursor.close()
            except Exception:
                pass
        if conn is not None:
            try:
                conn.close()
            except Exception:
                pass

In [38]:
upload_fs_data_to_db(fs_value_df, db_info)

업로드 대상 rows: 1,313,907
1,000 / 1,313,907 rows 완료
2,000 / 1,313,907 rows 완료
3,000 / 1,313,907 rows 완료
4,000 / 1,313,907 rows 완료
5,000 / 1,313,907 rows 완료
6,000 / 1,313,907 rows 완료
7,000 / 1,313,907 rows 완료
8,000 / 1,313,907 rows 완료
9,000 / 1,313,907 rows 완료
10,000 / 1,313,907 rows 완료
11,000 / 1,313,907 rows 완료
12,000 / 1,313,907 rows 완료
13,000 / 1,313,907 rows 완료
14,000 / 1,313,907 rows 완료
15,000 / 1,313,907 rows 완료
16,000 / 1,313,907 rows 완료
17,000 / 1,313,907 rows 완료
18,000 / 1,313,907 rows 완료
19,000 / 1,313,907 rows 완료
20,000 / 1,313,907 rows 완료
21,000 / 1,313,907 rows 완료
22,000 / 1,313,907 rows 완료
23,000 / 1,313,907 rows 완료
24,000 / 1,313,907 rows 완료
25,000 / 1,313,907 rows 완료
26,000 / 1,313,907 rows 완료
27,000 / 1,313,907 rows 완료
28,000 / 1,313,907 rows 완료
29,000 / 1,313,907 rows 완료
30,000 / 1,313,907 rows 완료
31,000 / 1,313,907 rows 완료
32,000 / 1,313,907 rows 완료
33,000 / 1,313,907 rows 완료
34,000 / 1,313,907 rows 완료
35,000 / 1,313,907 rows 완료
36,000 / 1,313,907 rows 완료
37,000 / 1,313

In [40]:
fs_value_df[fs_value_df['symbol'] == 'A000020']

,symbol,company_name,date,indicator,value
0,A000020,동화약품,2004-03-31,계속사업이익(천원),7.313530e+05
1,A000020,동화약품,2004-03-31,당기순이익(천원),7.313530e+05
2,A000020,동화약품,2004-03-31,당좌자산(천원),1.203545e+08
3,A000020,동화약품,2004-03-31,매출액(천원),3.254566e+07
4,A000020,동화약품,2004-03-31,매출채권및기타채권(천원),8.691194e+07
...,...,...,...,...,...
2176,A000020,동화약품,2025-12-31,당기순이익(천원),-1.733063e+06
2178,A000020,동화약품,2025-12-31,매출액(천원),1.236963e+08
2185,A000020,동화약품,2025-12-31,순이익률,-1.401063e-02
2187,A000020,동화약품,2025-12-31,영업이익(천원),-3.996403e+06


In [34]:
db_info = {
    "user": 'stox7412',
    "password": 'Apt106503!~',
    "host": '192.168.0.230',
    # 'host': 'hystox74.synology.me',
    "port": 3307,
    "database": "investar"
}

upload_fs_data_to_db(fs_value_df, db_info)

OperationalError: (1054, "Unknown column 'symbol' in 'field list'")